# Supplementary Table S6: Fixed-effect multinomial logit regression for 1990-2017

This notebook fits the multinomial logit models with temporal fixed-effects and prints the LaTeX regression table, followed by the Wald tests comparing the positive and negative coefficients.

It reads the processed data in the data folder (`path2data`) and can be run on its own.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns

from vizutils import pvalue2stars, table_row, make_multinomial_latex_table, logistic_coefficient_plot

import pyscisci.all as pyscisci

# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')


Please install python-Levenshtein to take full advantage of the NLP tools. 
 pip install python-Levenshtein
Please install sparse_dot_topn to take full advantage of the NLP tools. 
 pip install sparse_dot_topn


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


 # Analysis of the International Recognition Network

In [2]:
# lets focus on the citation without the same organization and self citations

country_auc_df = pd.read_csv(os.path.join(path2data, 'oa_countrycites_nosameorg_auc.csv.gz'))

country_auc_df = country_auc_df[country_auc_df['N']>=50]

country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5)/np.sqrt(country_auc_df['Cov'])
# one-sided test in the direction of the observed preference: H1 is AUC > 0.5 (over-citation)
# when Z > 0, and AUC < 0.5 (under-citation) when Z < 0
country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

def reject_holm(pvalues):
    reject, pvalues, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
    return reject
country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

net_auc = country_auc_df[country_auc_df['CitingCountry'] != country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
net_auc = net_auc[net_auc['Significant']]
self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
self_auc = self_auc[self_auc['Significant']]

syear, eyear = 1990, 2017
net_auc = net_auc[net_auc['CitedYear'] >= syear]
net_auc = net_auc[net_auc['CitedYear'] <= eyear]

comm2country = {0:['AE', 'BD', 'CN', 'CY', 'DZ', 'EG', 'ID', 'IN', 'IQ', 'IR', 'JO', 'KH', 'KR', 'KW', 'MA', 'MY', 'OM', 'PK', 'QA', 'SA', 'SG', 'TH', 'TN', 'TR', 'TW', 'VN'],
1:['AR', 'BN', 'BO', 'BR', 'BW', 'BY', 'CL', 'CM', 'CO', 'CR', 'GE', 'HK', 'HR', 'LU', 'MZ', 'NC', 'PA', 'PE', 'RU', 'SJ', 'ST', 'UY', 'VE'],
2:['AT', 'BE', 'CA', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HU', 'IE', 'IL', 'IS', 'IT', 'JP', 'MX', 'NL', 'NO', 'NP', 'NZ', 'PH', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK'],
3:['AU', 'CH', 'GB', 'US'],
4:['BH', 'ET', 'GH', 'KE', 'MW', 'NG', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']}

country2comm = {c:ci for ci,clist in comm2country.items() for c in clist}
poscountrylist = sorted(list(country2comm.keys()))

connected_countries = np.sort(np.unique(net_auc[net_auc['Zscore'] > 0][['CitingCountry', 'CitedCountry']].values.flatten()))

# Multinomial Link Prediction

In [3]:
panel_data = pd.read_csv(os.path.join(path2data, 'link_prediction_panel_data_1990_2017.csv.gz'))


net_panel = panel_data.merge(net_auc.rename(columns={'CitingCountry':'oCountry', 'CitedCountry':'tCountry','CitedYear':'year'}), 
                           how='left', on=['oCountry','tCountry','year'])

net_panel['edge_preference'] = 1 + (net_panel['Zscore'] < 0).astype(int)
net_panel.loc[net_panel['Significant'].isnull(), 'edge_preference'] = 0

# the citation preferences cover 1990-2017, so the panel rows of later years are not used
net_panel = net_panel[(net_panel['year'] >= syear) & (net_panel['year'] <= eyear)].reset_index(drop=True)

net_panel['edge_preference'].unique()

dummy_years=['year_'+str(i) for i in range(syear + 1, eyear + 1)]

all_dummies=['comlang_off','SameContinent']+dummy_years

net_panel_with_dummies = pd.get_dummies(net_panel, columns=['year'], drop_first=True)
net_panel_with_dummies[dummy_years] = net_panel_with_dummies[dummy_years].astype(int)



v1=['oGDP_PCAP','tGDP_PCAP', 'oPop','tPop']
v2=['PhysicalDistance','SameContinent']
v3=['comlang_off','TopicSimilarity']
v4=['cumulative_laws']
v5=['CollabStrength']
v6=['oTopFrac','tTopFrac']


all_model_features = v1 + v2 + v3 + v4 + v5 + v6

# continuous variables are standardized once over the 1990-2017 panel, so all models share the same scale
for c in all_model_features:
    if net_panel_with_dummies[c].nunique() > 2:
        net_panel_with_dummies[c] = (net_panel_with_dummies[c] - net_panel_with_dummies[c].mean())/net_panel_with_dummies[c].std()

exog_var_sets = [v1+dummy_years,v2,v3,v4,v5,v6]
dep_var = "edge_preference"

Nmodels = len(exog_var_sets)

fit_models = []
for icol in range(Nmodels):
    exog_vars = list(chain.from_iterable(exog_var_sets[:(icol+1)]))
    X = net_panel_with_dummies[exog_vars + [dep_var]].dropna().reset_index(drop=True)

    Y = X[dep_var]
    X = sm.add_constant(X[exog_vars])

    model = sm.MNLogit(Y, X).fit()
    fit_models.append(model)
    

Optimization terminated successfully.
         Current function value: 0.039011
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.039781
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.038455
         Iterations 13
Optimization terminated successfully.
         Current function value: 0.038375
         Iterations 13
Optimization terminated successfully.
         Current function value: 0.036143
         Iterations 13
Optimization terminated successfully.
         Current function value: 0.034350
         Iterations 13


In [4]:
%load_ext autoreload
%autoreload 2

In [5]:

from vizutils import pvalue2stars, table_row, make_multinomial_latex_table, logistic_coefficient_plot

namedict = {'const':'Intercept', 'oTopFrac':'Origin top journal fraction', 'tTopFrac':'Target top journal fraction', 
'oGDP_PCAP':'Log origin GDP per capita', 'oPop':'Log origin population','tPop':'Log target population',
           'tGDP_PCAP':'Log target GDP per capita', 'PhysicalDistance': 'Physical distance', 'TopicSimilarity':'Field similarity', 
            'comlang_off':'Same official language', 'CollabStrength':'Log collaboration strength','SameContinent':'Same continent',
           'cumulative_laws':'Bilateral research agreements'}
years={i: i[-4:] for i in dummy_years}
merged_namedict = years.copy()  # Create a copy of dict1 to avoid modifying the original
merged_namedict.update(namedict)

# the LaTeX tables are saved in the tables folder
path2tables = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'tables')
os.makedirs(path2tables, exist_ok=True)

make_multinomial_latex_table(fit_models, exog_var_sets, dep_var, merged_namedict,
                             caption_text="Fixed-effect multinomial logit regression for 1990-2017.",
                             label="table:multinomialfull", fname=os.path.join(path2tables, "TableS6.tex"))


{\tiny
    \begin{longtable}{p{0.2\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}}\caption{\textbf{Fixed-effect multinomial logit regression for 1990-2017.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\chi^2$ is a likelihood-ratio test against the intercept-only model.} 
      \label{table:multinomialfull} \\
      \hline \hline \\
    \multicolumn{7}{c}{\textbf{Dependent variable: Citation preference}} \\ \hline 
     & \multicolumn{6}{c}{Model}  \\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\[0.8ex]
    \hline
    \endfirsthead\multicolumn{2}{c}%
      {{\tablename\ \thetable{} -- continued from previous page}} \\
      \hline \\& (1)& (2

'{\\tiny\n    \\begin{longtable}{p{0.2\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}}\\caption{\\textbf{Fixed-effect multinomial logit regression for 1990-2017.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\\chi^2$ is a likelihood-ratio test against the intercept-only model.} \n      \\label{table:multinomialfull} \\\\\n      \\hline \\hline \\\\\n    \\multicolumn{7}{c}{\\textbf{Dependent variable: Citation preference}} \\\\ \\hline \n     & \\multicolumn{6}{c}{Model}  \\\\\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\\\[0.8ex]\n    \\hline\n    \\endfirsthead\\multicolumn{2}{c}%\n      {{\\tablename\\ \\thetable{} -- continued fr

The same table split into two tables that each fit on one page: TableS6a (positive citation preference) and TableS6b (negative citation preference and the model statistics), written to `tables/TableS6_split.tex`.

In [6]:
from vizutils import make_multinomial_latex_tables_split

make_multinomial_latex_tables_split(fit_models, exog_var_sets, dep_var, merged_namedict,
                                    caption_text="Fixed-effect multinomial logit regression for 1990-2017.",
                                    label="table:multinomialfull", fname=os.path.join(path2tables, "TableS6_split.tex"))

\refstepcounter{table}\label{table:multinomialfull}\addtocounter{table}{-1}%
\begin{table}[H]
\let\origthetable\thetable\renewcommand{\thetable}{\origthetable a}
\caption{\textbf{Fixed-effect multinomial logit regression for 1990-2017. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \ref{table:multinomialfull:negative}.}
\label{table:multinomialfull:positive}
\centering
{\tiny
\begin{tabular}{p{0.2\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}}
\hline \hline \\[-1.8ex]
\multicolumn{7}{c}{\textbf{Dependent variable: Citation preference (p

'\\refstepcounter{table}\\label{table:multinomialfull}\\addtocounter{table}{-1}%\n\\begin{table}[H]\n\\let\\origthetable\\thetable\\renewcommand{\\thetable}{\\origthetable a}\n\\caption{\\textbf{Fixed-effect multinomial logit regression for 1990-2017. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \\ref{table:multinomialfull:negative}.}\n\\label{table:multinomialfull:positive}\n\\centering\n{\\tiny\n\\begin{tabular}{p{0.2\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}}\n\\hline \\hline \\\\[-1.8ex]\n\\multicolumn{7}{c}{\\textbf{D

## Wald tests of the positive vs negative coefficients

In [7]:
def summary_table(model,ycat,ycat_name):
    
    dfpo=model.params.reset_index().rename(columns={'index':'item'})
    dfc1=dfpo[['item',ycat-1]].rename(columns={ycat-1:'risk'})
    
    dfpv=model.pvalues.reset_index().rename(columns={'index':'item'})
    dfpv=dfpv[['item',ycat-1]].rename(columns={ycat-1:'pvalues'})

    df1=dfc1.merge(dfpv,how='left',on='item')
    
    dfcon1=model.conf_int().loc[str(ycat)].reset_index().rename(columns={'index':'item','lower':0,'upper':1})
    df2=df1.merge(dfcon1,how='left',on='item')

    df2['strf']=ycat_name

    return df2

dfc1=summary_table(fit_models[-1],1,'Positive recognition')
dfc2=summary_table(fit_models[-1],2,'Negative recognition')

dfs=pd.concat([dfc1,dfc2])

namedict = {'const':'Intercept', 
            'oTopCumuFrac':'Source top journal fraction', 'tTopCumuFrac':'Target top journal fraction',
            'oPop':'Source population','tPop':'Target population',
             'oGDP_PCAP':'Source GDP per capita', 'tGDP_PCAP':'Target GDP per capita',
            'oTopFrac':'Source top journal fraction', 'tTopFrac':'Target top journal fraction',
            'PhysicalDistance': 'Physical distance', 'tradeflow_baci':'Trade volume', 
            'FieldDistance':'Topic distance', 'SameContinent':'Same continent',
            'CollabStrength':'Collaboration strength',
            'CosineNormStrength':'Normed collaboration strength',
            'comlang_off':'Same official language', 
            'DemocDis': 'Democracy distance','oDemoc':'Source democracy', 'tDemoc':'Target democracy',
           'oGNI_PCAP':'Source GNI per capita', 'tGNI_PCAP':'Target GNI per capita',
           'GNI_PCAP':'GNI per capita', 'GNI':'GNI', 'GDP_PCAP':'GDP per capita',
           'Pop':'Population', 'NumPub':'Number of publications','Agreements':'Bilateral research agreement',
           'RND_per':'Percent of GDP for R&D', "FractionNationalAuthors":"Fraction national authors"}

dfs.replace(namedict,inplace=True)

dfs['category']=dfs['item']

#drop the year fixed effect rows
dfs = dfs[~dfs['item'].str.contains('year')].reset_index(drop=True)

df1 = dfs[dfs["strf"] == "Positive recognition"].reset_index(drop=True)
df2 = dfs[dfs["strf"] == "Negative recognition"].reset_index(drop=True)

df1['xerror']=df1[1]-df1['risk']
df2['xerror']=df2[1]-df2['risk']

df1=df1[df1['item']!='Intercept']
df2=df2[df2['item']!='Intercept']

#dfs

In [8]:
def wald_mlogit_diff(model, param1, param2=None):
    covar = model.cov_params()
    flipvalue = {'1':'2', '2':'1'}
    if param2 is None:
        param2 = ":".join([flipvalue.get(p, p) for p in param1.split(':')])

    pidx1 = tuple(param1.split(':'))[::-1]
    pidx2 = tuple(param2.split(':'))[::-1]

    diffse = np.sqrt( covar[pidx1][pidx1] + covar[pidx2][pidx2] - 2*covar[pidx1][pidx2] )

    coeffdiff = model.params[int(pidx1[0])-1][pidx1[1]] - model.params[int(pidx2[0])-1][pidx2[1]]
    zscore = coeffdiff/diffse
    pvalue = spstats.norm.sf(np.abs(zscore))*2
    
    return zscore, pvalue

for it in dfc1['item'].values:
    if it != 'const':
        param1 = it+':1'
        print(merged_namedict[it], wald_mlogit_diff(fit_models[-1], param1))


Log origin GDP per capita (-26.33760516637262, 7.117600808985407e-153)
Log target GDP per capita (-36.917699700695806, 2.4033588550120813e-298)
Log origin population (-31.19455875756443, 1.2627813621624332e-213)
Log target population (-30.497880390166873, 2.7800920291316967e-204)
1991 (-0.10764840036944551, 0.9142746020931766)
1992 (-0.5463375477621675, 0.5848339306837338)
1993 (0.477518685671603, 0.6329928182650597)
1994 (-0.09764906202688725, 0.9222109647704265)
1995 (-0.3427132610122601, 0.7318141854169172)
1996 (-0.790898037730765, 0.4290034935699718)
1997 (-0.9183988656270444, 0.3584100843841699)
1998 (-0.9384535952973355, 0.34801135398989513)
1999 (-1.1287012416233786, 0.25902388535084764)
2000 (-0.1848585145039418, 0.8533400107387383)
2001 (-0.6954553317656618, 0.4867699950149631)
2002 (-0.5745472941883392, 0.5655975062656198)
2003 (-0.06183631044813064, 0.9506931872476319)
2004 (0.31193300368793214, 0.7550914397133157)
2005 (0.673432002624307, 0.5006724945345586)
2006 (0.828456

## Source Data for Supplementary Table 6

The values reported in the table, written to `Source Data/SourceData_6_TableS6.xlsx`.

In [9]:
# the Source Data helpers (sourcedatautils.py, in the code folder like vizutils.py)
from sourcedatautils import write_source_data, path2sourcedata, path2data

title = 'Supplementary Table 6: Fixed-effect multinomial logit regression for 1990-2017.'
description = ('Coefficients of the multinomial logit models (1)-(6) with temporal fixed-effects predicting a positive '
               'or negative citation preference compared to no preference (baseline). Continuous variables are '
               'standardized once over the 1990-2017 country-pair panel. The year fixed effects are included in all models '
               'but not shown.')

coefs = []
fits = []
for imodel, model in enumerate(fit_models):
    conf = model.conf_int()
    for icat, outcome in [(1, 'Positive'), (2, 'Negative')]:
        df = pd.DataFrame({'Coefficient':model.params[icat-1], 'SE':model.bse[icat-1], 'p-value':model.pvalues[icat-1]})
        df['CI lower (95%)'] = conf.loc[str(icat)]['lower']
        df['CI upper (95%)'] = conf.loc[str(icat)]['upper']
        df = df[~df.index.str.startswith('year_')]
        df.insert(0, 'Variable', df.index.map(lambda v: merged_namedict.get(v, v)))
        df.insert(0, 'Citation preference', outcome)
        df.insert(0, 'Model', imodel + 1)
        coefs.append(df[['Model', 'Citation preference', 'Variable', 'Coefficient', 'CI lower (95%)', 'CI upper (95%)',
                         'SE', 'p-value']])
    fits.append([imodel + 1, int(model.nobs), model.prsquared, model.llf, model.llr, model.df_model, model.llr_pvalue])

coefs = pd.concat(coefs, ignore_index=True)
fits = pd.DataFrame(fits, columns=['Model', 'Observations', 'Pseudo R2', 'Log likelihood', 'LLR chi2', 'd.f.',
                                   'LLR p-value'])

tables = [('Coefficients', 'Coefficients of each model for the positive and negative citation preference.', coefs),
          ('Model fit', 'Observations and goodness of fit of each model.', fits)]

write_source_data('SourceData_6_TableS6.xlsx', title, description, tables)

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_6_TableS6.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
